# **ginger**

In [1]:
import sys

if "google.colab" in sys.modules:
    import getpass

    token = getpass.getpass()

    !git clone https://{token}@github.com/Mango-Cats/ginger.git
    %cd ginger

    %load_ext autoreload
    %autoreload 2

    !pip install -e . --no-deps

In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.11.0+cu130
cpu


## **Data Loading & Preprocessing**

In [3]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [4]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")


Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [5]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [6]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)

/home/zrgnt/ginger/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


## **Training**

In [10]:
from src.trainer.evaluate import evaluate
from src.trainer.train import train_epoch

In [12]:
train_losses = []
val_losses = []

print(f"\nStarting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model=model, dataloader=train_loader, criterion=CRITERION, optimizer=OPTIMIZER, scheduler=SCHEDULER, device=DEVICE)
    val_loss = evaluate(model, val_loader, CRITERION, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


Starting training for 100 epochs...



/home/zrgnt/ginger/.venv/lib/python3.11/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


KeyboardInterrupt: 

## **Inference**

In [ ]:
import random
from src.architecture.inference import greedy_decode
from src.evaluation.metrics import compute_metrics

In [ ]:
test_pairs = test_loader.dataset.data_pairs
predictions = []
references = []

for i, (graphemes, true_phonemes) in enumerate(test_pairs):
    word = ''.join(graphemes)
    
    predicted_phonemes, _ = greedy_decode(model, word, grapheme_vocab, phoneme_vocab, DEVICE)
    
    predictions.append(predicted_phonemes)
    references.append(true_phonemes)

NameError: name 'compute_metrics' is not defined

In [ ]:
per, wer = compute_metrics(predictions, references)
total_words = len(test_pairs)
print("\nMetrics:")
print(f"Total Words Tested: {total_words}")
print(f"Phoneme Error Rate (PER): {per:.2f}%")
print(f"Word Error Rate (WER):    {wer:.2f}%")

print("\nSample Predictions:")
sample_indices = random.sample(range(total_words), 5)
for idx in sample_indices:
    grap = ''.join(test_pairs[idx][0])
    print(f"\nWord:     {grap.upper()}")
    print(f"Expected: {' '.join(references[idx])}")
    print(f"Output:   {' '.join(predictions[idx])}")


Metrics:
Total Words Tested: 1824
Phoneme Error Rate (PER): 86.31%
Word Error Rate (WER):    99.78%

Sample Predictions:

Word:     J
Expected: h o t a
Output:   h e ɾ j o

Word:     KALMOT
Expected: k a l m o t
Output:   m a k t a l o k t

Word:     BISIKLETA
Expected: b i s i k l e t a
Output:   b i l i k a s t e s t i l i k a

Word:     MANGUTANG
Expected: m a ŋ u t a ŋ
Output:   m a ŋ ɡ u n t a ŋ

Word:     JACINTO
Expected: h a s i n t o
Output:   t i n a k o n t i a
